In [2]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()
sns.set_theme(style='whitegrid')

In [7]:
def get_table(ds_name, allocation_scheme, seeds, methods, test=True, ratio=False):
    h = hyperparams[ds_name][allocation_scheme]
    method_full_names = {'FedAvg': 'FedAvg', 'qFFL': f'qFFL_q{h["qFFL"]}', 'PropFair': f'PropFair_base{h["PropFair"]}',
                   'FedMGDA': f'FedMGDA_epsilon{h["FedMGDA"]}', 'FedFV': f'FedFV_alpha{h["FedFV"][0]}_tau{h["FedFV"][1]}',
                   'Tche': 'Tche', 'STche': f'STche_gamma{h["STche"]}',
                   'STche_momentum': f'STche_gamma{h["STche_momentum"]}_momentum',
                   'AFL': f'AFL_llr{h["AFL"]}', 'AFL_new': f'AFL_new_llr{h["AFL_new"]}', 
                   'AFLeg': f'AFLeg_llr{h["AFLeg"]}', 'AFLeg_new': f'AFLeg_new_llr{h["AFLeg_new"]}',
                   'ExcessMTL': f'ExcessMTL_llr{h["ExcessMTL"]}', 'AdaExcessMTL': f'AdaExcessMTL_llr{h["AdaExcessMTL"]}',
                   'EPO': 'EPO', 'FERERO': 'FERERO',
                   'AFL_new_train': f'AFL_new_llr{h["AFL_new"]}_train',
                   'AFL_new_val': f'AFL_new_llr{h["AFL_new"]}_val',
                   'AFLeg_new_train': f'AFLeg_new_llr{h["AFLeg_new"]}_train',
                   'AFLeg_new_val': f'AFLeg_new_llr{h["AFLeg_new"]}_val',
                   'Tche_ratio': 'Tche_ratio',
                   'STche_ratio': f'STche_gamma{h["STche"]}_ratio',
                   'AFL_ratio': f'AFL_llr{h["AFL"]}_ratio',
                   'AFL_new_ratio': f'AFL_new_llr{h["AFL_new"]}_ratio',
                   'AFLeg_ratio': f'AFLeg_llr{h["AFLeg"]}_ratio',
                   'AFLeg_new_ratio': f'AFLeg_new_llr{h["AFLeg_new"]}_ratio',}
    method_full = [method_full_names[method] for method in methods]

    metrics = ['Average accuracy', 'Agnostic loss', 'Accuracy parity']
    data = [[[] for i in range(len(methods))] for _ in range(len(metrics))]

    for i, method in enumerate(method_full):
        for seed in seeds:
            path = f'../output/10seeds/{ds_name}_{allocation_scheme}/seed{seed}/{method}/log.pickle'
            with open(path, 'rb') as file:
                res = pickle.load(file)
            assert res[-1]['round'] == 299 if ds_name == 'MNIST' else 44, f"Wrong number of epochs for {method}: {res[-1]['round']}!"
            if test:
                if ratio == False:
                    data[0][i].append(res[-1]['test_infer_stats']['mean_acc'] * 100)
                    data[1][i].append(np.max(res[-1]['test_infer_stats']['losses']))
                    data[2][i].append(np.sqrt(np.var(res[-1]['test_infer_stats']['accuracys']) * 10000))
                else:
                    data[0][i].append(res[-1]['test_infer_stats']['mean_acc'] * 100)
                    final_losses = res[-1]['test_infer_stats']['losses']
                    init_losses = res[0]['test_infer_stats']['losses']
                    data[1][i].append(np.max([final / init for (final, init) in zip(final_losses, init_losses)]))
                    data[2][i].append(np.sqrt(np.var([final / init for (final, init) in zip(final_losses, init_losses)])))
            else:
                data[0][i].append(res[-1]['train_infer_stats']['mean_acc'] * 100)
                data[1][i].append(np.max(res[-1]['train_infer_stats']['losses']))
                data[2][i].append(np.sqrt(np.var(res[-1]['train_infer_stats']['accuracys']) * 10000))

    
    avg = np.zeros((len(metrics), len(method_full)))
    std = np.zeros((len(metrics), len(method_full)))
    for i, method in enumerate(method_full):
        for j in range(len(metrics)):
            avg[j][i] = np.mean(data[j][i])
            std[j][i] = np.std(data[j][i])

    # Create DataFrame for avg values (optional for convenience)
    df_avg = pd.DataFrame(avg, index=metrics, columns=methods)

    separator = '\t\t'
    # Print the header (column names) with the custom separator
    header = separator.join(df_avg.columns)
    print(f'Index\t{header}')

    # Manually format and print each row with the custom separator
    for j, metric in enumerate(metrics):
        formatted_row = separator.join(f'{avg_val:.3f}±{std_val:.3f}' for avg_val, std_val in zip(avg[j], std[j]))
        print(f'{metric}\t{formatted_row}')
    
    if avg.shape[1] > 1:
        for j, metric in enumerate(metrics):
            if j == 0:
                print(metric, np.sort(avg[j])[-1], np.sort(avg[j])[-2])
            else:
                print(metric, np.sort(avg[j])[0], np.sort(avg[j])[1])

In [4]:
# 10 seeds
hyperparams = {
    'MNIST': {
        'rotation': {
            'AFL': 0.03, 'AFLeg': 0.3, 'AFL_new': 0.03, 'AFLeg_new': 0.1,
            'STche': 0.01, 'STche_momentum': None, 'qFFL': 0.1, 'FedMGDA': 0.5, 'FedFV': [0.2, 1], 'PropFair': 2.0,
            'ExcessMTL': 1.0, 'AdaExcessMTL': 0.3
        },
        'partial_class_ni2': {
            'AFL': 0.01, 'AFLeg': 0.3, 'AFL_new': 1.0, 'AFLeg_new': 0.3,
            'STche': 0.01, 'STche_momentum': None, 'qFFL': 0.1, 'FedMGDA': 0.5, 'FedFV': [0.2, 1], 'PropFair': 2.0,
            'ExcessMTL': 1.0, 'AdaExcessMTL': 1.0
        },
        'partial_class_ni5': {
            'AFL': 0.01, 'AFLeg': 0.1, 'AFL_new': 1.0, 'AFLeg_new': 0.03,
            'STche': 0.01, 'STche_momentum': None, 'qFFL': 0.1, 'FedMGDA': 0.05, 'FedFV': [0.2, 1], 'PropFair': 2.0,
            'ExcessMTL': 1.0, 'AdaExcessMTL': 1.0
        }
    },
    'CIFAR10': {
        'rotation': {
            'AFL': 1.0, 'AFLeg': 1.0, 'AFL_new': 0.03, 'AFLeg_new': 0.3,
            'STche': 0.01, 'STche_momentum': 0.01, 'qFFL': 0.1, 'FedMGDA': 0.1, 'FedFV': [0.2, 1], 'PropFair': 3.0,
            'ExcessMTL': 1.0, 'AdaExcessMTL': 0.1
        },
        'partial_class_ni2': {
            'AFL': 0.01, 'AFLeg': 0.03, 'AFL_new': 0.003, 'AFLeg_new': 0.03,
            'STche': 0.03, 'STche_momentum': 0.03, 'qFFL': 0.1, 'FedMGDA': 0.1, 'FedFV': [0.1, 0], 'PropFair': 4.0,
            'ExcessMTL': 0.1, 'AdaExcessMTL': 0.03
        },
        'partial_class_ni5': {
            'AFL': 0.01, 'AFLeg': 0.1, 'AFL_new': 0.3, 'AFLeg_new': 0.03,
            'STche': 0.03, 'STche_momentum': 0.01, 'qFFL': 0.1, 'FedMGDA': 0.1, 'FedFV': [0.2, 1], 'PropFair': 2.0,
            'ExcessMTL': 0.1, 'AdaExcessMTL': 0.001
        }
    }
}

In [5]:
methods = ['FedAvg', 'qFFL', 'PropFair', 'FedMGDA', 'FedFV', 'Tche', 'STche', 'EPO', 'FERERO', 'ExcessMTL', 'AdaExcessMTL', 'AFL', 'AFL_new', 'AFLeg', 'AFLeg_new']
seeds = [0, 25, 37, 42, 53, 81, 119, 1010, 1201, 2003]

In [6]:
get_table('MNIST', 'rotation', seeds, methods)

Index	FedAvg		qFFL		PropFair		FedMGDA		FedFV		Tche		STche		EPO		FERERO		ExcessMTL		AdaExcessMTL		AFL		AFL_new		AFLeg		AFLeg_new
Average accuracy	92.450±0.234		91.896±0.226		90.622±0.194		92.416±0.210		94.501±0.179		92.579±0.270		92.977±0.230		92.604±0.302		92.443±0.232		88.566±0.279		92.520±0.235		88.597±0.241		92.593±0.236		88.664±0.220		92.583±0.185
Agnostic loss	0.639±0.028		0.675±0.027		0.742±0.025		0.322±0.013		0.302±0.027		0.342±0.027		0.409±0.027		0.350±0.029		0.640±0.028		0.503±0.013		0.326±0.013		0.514±0.024		0.341±0.019		0.518±0.026		0.334±0.019
Accuracy parity	4.796±0.272		5.021±0.262		5.454±0.152		1.153±0.168		1.646±0.335		1.397±0.292		2.385±0.247		1.467±0.288		4.807±0.271		1.637±0.106		1.117±0.148		1.658±0.257		1.296±0.211		1.680±0.238		1.201±0.227
Average accuracy 94.50100401606427 92.97690763052209
Agnostic loss 0.3020831301808357 0.3220303446054459
Accuracy parity 1.116583860630441 1.1530095298417673


In [7]:
get_table('MNIST', 'partial_class_ni2', seeds, methods)

Index	FedAvg		qFFL		PropFair		FedMGDA		FedFV		Tche		STche		EPO		FERERO		ExcessMTL		AdaExcessMTL		AFL		AFL_new		AFLeg		AFLeg_new
Average accuracy	92.330±1.429		90.264±1.714		91.274±1.505		92.236±1.405		93.675±1.331		93.250±1.468		92.592±1.558		91.866±2.986		92.301±1.467		91.354±1.368		92.435±1.514		90.965±1.502		92.387±1.616		90.133±1.850		92.209±1.580
Agnostic loss	0.534±0.106		0.659±0.118		0.573±0.105		0.386±0.055		0.397±0.094		0.409±0.067		0.477±0.098		0.517±0.152		0.545±0.111		0.489±0.047		0.404±0.066		0.549±0.092		0.424±0.099		0.569±0.111		0.435±0.105
Accuracy parity	4.574±1.469		6.025±1.696		4.987±1.620		2.629±0.489		3.318±0.868		3.685±0.982		3.989±1.201		4.273±1.772		4.700±1.500		3.530±0.673		2.753±0.523		4.373±1.027		3.153±0.817		4.563±1.095		3.084±0.865
Average accuracy 93.67496909198351 93.24997017592544
Agnostic loss 0.38626953065395353 0.39738252460956575
Accuracy parity 2.629473207123635 2.7529949649225465


In [8]:
get_table('MNIST', 'partial_class_ni5', seeds, methods)

Index	FedAvg		qFFL		PropFair		FedMGDA		FedFV		Tche		STche		EPO		FERERO		ExcessMTL		AdaExcessMTL		AFL		AFL_new		AFLeg		AFLeg_new
Average accuracy	93.907±0.418		93.008±0.514		92.865±0.462		93.944±0.480		95.017±0.515		94.690±0.317		94.140±0.405		93.402±0.942		93.891±0.421		92.424±0.460		94.063±0.444		92.256±0.501		94.239±0.440		92.261±0.505		94.021±0.432
Agnostic loss	0.282±0.034		0.320±0.034		0.328±0.030		0.267±0.027		0.231±0.032		0.250±0.045		0.272±0.032		0.304±0.047		0.283±0.034		0.342±0.022		0.264±0.031		0.349±0.025		0.263±0.033		0.352±0.028		0.275±0.032
Accuracy parity	1.559±0.378		1.752±0.431		1.727±0.424		1.175±0.244		1.287±0.306		1.518±0.375		1.360±0.335		1.910±0.484		1.564±0.403		1.578±0.443		1.205±0.268		1.733±0.445		1.193±0.280		1.799±0.482		1.351±0.371
Average accuracy 95.01686755154492 94.69031403148321
Agnostic loss 0.23143647015094757 0.24967894256114959
Accuracy parity 1.1754927175328214 1.1933863911474973


In [9]:
get_table('CIFAR10', 'rotation', seeds, methods)

Index	FedAvg		qFFL		PropFair		FedMGDA		FedFV		Tche		STche		EPO		FERERO		ExcessMTL		AdaExcessMTL		AFL		AFL_new		AFLeg		AFLeg_new
Average accuracy	66.269±0.541		61.514±0.450		63.108±0.705		66.321±0.264		66.242±0.475		62.556±1.546		66.320±0.403		61.380±2.680		66.368±0.355		59.322±0.520		66.211±0.412		59.244±0.489		66.218±0.467		59.273±0.443		65.885±0.610
Agnostic loss	1.260±0.044		1.314±0.041		1.286±0.044		1.197±0.035		1.201±0.038		1.266±0.066		1.219±0.043		1.293±0.062		1.259±0.040		1.256±0.028		1.227±0.041		1.239±0.029		1.148±0.037		1.254±0.033		1.156±0.045
Accuracy parity	6.012±0.596		5.451±0.514		5.312±0.385		4.632±0.430		4.668±0.581		4.727±1.099		4.997±0.712		4.731±0.928		5.922±0.665		2.795±0.332		5.615±0.745		2.429±0.531		3.592±0.487		2.660±0.549		3.688±0.646
Average accuracy 66.36800000000001 66.321
Agnostic loss 1.1483205127716065 1.156049989461899
Accuracy parity 2.4292991149253265 2.6599499392558155


In [8]:
get_table('CIFAR10', 'rotation', seeds, ['STche_momentum'])

Index	STche_momentum
Average accuracy	66.422±0.526
Agnostic loss	1.205±0.027
Accuracy parity	4.890±0.451


In [ ]:
methods = ['AFL_new', 'AFL_new_train', 'AFL_new_val', 'AFLeg_new', 'AFLeg_new_train', 'AFLeg_new_val']
seeds = [0, 25, 37, 42, 53, 81, 119, 1010, 1201, 2003]
get_table('CIFAR10', 'rotation', seeds, methods)

Index	AFL_new		AFL_new_train		AFL_new_val		AFLeg_new		AFLeg_new_train		AFLeg_new_val
Average accuracy	66.218±0.467		66.061±0.368		66.386±0.435		65.885±0.610		65.913±0.160		66.260±0.716
Agnostic loss	1.148±0.037		1.128±0.036		1.185±0.057		1.156±0.045		1.130±0.037		1.171±0.051
Accuracy parity	3.592±0.487		3.503±0.759		4.022±0.561		3.688±0.646		3.372±0.578		3.819±0.602
Average accuracy 66.386 66.25999999999999
Agnostic loss 1.1280854362249375 1.1299870365858078
Accuracy parity 3.371817471791458 3.5028183475840273


In [ ]:
get_table('CIFAR10', 'rotation', seeds, 
          ['FedAvg', 'Tche_ratio', 'STche_ratio', 'AFL_ratio', 'AFL_new_ratio', 'AFLeg_ratio', 'AFLeg_new_ratio'],
          ratio=True)

Index	FedAvg		Tche_ratio		STche_ratio		AFL_ratio		AFL_new_ratio		AFLeg_ratio		AFLeg_new_ratio
Average accuracy	66.269±0.541		61.999±1.766		66.222±0.397		59.197±0.440		66.046±0.520		59.177±0.439		66.130±0.608
Agnostic loss	0.656±0.021		0.648±0.029		0.627±0.021		0.647±0.013		0.465±0.019		0.651±0.018		0.465±0.022
Accuracy parity	0.087±0.011		0.066±0.010		0.068±0.010		0.024±0.006		0.043±0.006		0.029±0.006		0.042±0.007
Average accuracy 66.269 66.22200000000001
Agnostic loss 0.4648647988020206 0.4648839885656164
Accuracy parity 0.02435413201709202 0.028548081199291402


In [10]:
get_table('CIFAR10', 'partial_class_ni2', seeds, methods)

Index	FedAvg		qFFL		PropFair		FedMGDA		FedFV		Tche		STche		EPO		FERERO		ExcessMTL		AdaExcessMTL		AFL		AFL_new		AFLeg		AFLeg_new
Average accuracy	38.925±2.325		34.995±2.658		35.505±2.107		38.560±2.974		39.015±3.170		20.935±6.355		35.380±3.949		20.510±7.571		38.700±3.236		35.240±2.402		38.330±2.893		31.080±4.816		36.765±3.475		34.865±2.422		36.755±3.373
Agnostic loss	2.477±0.256		2.584±0.255		2.486±0.215		2.148±0.089		2.394±0.197		4.378±0.387		2.386±0.228		4.364±0.641		2.468±0.260		2.526±0.294		2.434±0.277		2.515±0.269		2.372±0.239		2.534±0.244		2.388±0.259
Accuracy parity	21.801±5.071		23.085±5.014		22.642±4.907		14.227±2.428		19.289±4.173		27.199±3.398		19.139±6.063		24.116±4.796		21.922±4.676		22.256±5.647		21.518±5.055		20.600±6.157		18.422±5.435		22.045±5.490		18.933±5.478
Average accuracy 39.015 38.925
Agnostic loss 2.147933727502823 2.372427773475647
Accuracy parity 14.227141960670846 18.422021998554786


In [9]:
get_table('CIFAR10', 'partial_class_ni2', seeds, ['STche_momentum'])

Index	STche_momentum
Average accuracy	35.105±4.236
Agnostic loss	2.380±0.234
Accuracy parity	18.945±5.735


In [11]:
get_table('CIFAR10', 'partial_class_ni5', seeds, methods)

Index	FedAvg		qFFL		PropFair		FedMGDA		FedFV		Tche		STche		EPO		FERERO		ExcessMTL		AdaExcessMTL		AFL		AFL_new		AFLeg		AFLeg_new
Average accuracy	55.626±1.606		49.316±1.859		53.794±1.530		56.174±1.339		55.746±1.448		36.242±4.324		55.328±1.656		37.458±2.333		55.662±1.277		48.142±1.862		55.766±1.753		47.960±1.874		53.422±1.402		48.064±1.952		55.632±1.524
Agnostic loss	1.705±0.125		1.856±0.147		1.740±0.137		1.596±0.143		1.662±0.098		3.748±0.279		1.622±0.048		3.138±0.421		1.729±0.157		1.870±0.136		1.705±0.170		1.832±0.080		1.664±0.065		1.833±0.089		1.662±0.074
Accuracy parity	8.610±1.751		9.614±2.169		9.037±1.697		6.862±1.842		7.609±1.284		16.141±2.355		7.311±1.253		14.602±1.629		8.854±2.064		9.715±2.075		8.354±2.072		8.787±1.413		6.823±1.504		8.803±1.133		7.930±1.185
Average accuracy 56.174 55.766
Agnostic loss 1.5960888695716857 1.6221975207328796
Accuracy parity 6.822589921526239 6.86218584506004


In [10]:
get_table('CIFAR10', 'partial_class_ni5', seeds, ['STche_momentum'])

Index	STche_momentum
Average accuracy	54.240±1.420
Agnostic loss	1.637±0.062
Accuracy parity	7.335±1.572


### training data

In [12]:
get_table('MNIST', 'rotation', seeds, methods, test=False)

Index	FedAvg		qFFL		PropFair		FedMGDA		FedFV		Tche		STche		EPO		FERERO		ExcessMTL		AdaExcessMTL		AFL		AFL_new		AFLeg		AFLeg_new
Average accuracy	93.309±0.116		92.565±0.140		91.087±0.125		93.347±0.101		96.572±0.066		93.567±0.280		93.986±0.090		93.589±0.229		93.306±0.117		88.582±0.186		93.461±0.124		88.622±0.140		93.591±0.109		88.671±0.140		93.554±0.109
Agnostic loss	0.598±0.012		0.642±0.014		0.724±0.013		0.263±0.005		0.159±0.003		0.255±0.012		0.337±0.003		0.262±0.010		0.599±0.012		0.466±0.010		0.259±0.005		0.470±0.010		0.246±0.004		0.475±0.010		0.246±0.003
Accuracy parity	4.417±0.152		4.680±0.166		5.071±0.186		0.653±0.074		0.579±0.063		0.457±0.163		1.432±0.071		0.399±0.138		4.433±0.155		0.661±0.144		0.502±0.069		0.573±0.153		0.253±0.050		0.645±0.130		0.280±0.059
Average accuracy 96.57188125416944 93.98599066044031
Agnostic loss 0.15858468413352966 0.2459363043308258
Accuracy parity 0.25293559767766627 0.28008089321999624


In [13]:
get_table('MNIST', 'partial_class_ni2', seeds, methods, test=False)

Index	FedAvg		qFFL		PropFair		FedMGDA		FedFV		Tche		STche		EPO		FERERO		ExcessMTL		AdaExcessMTL		AFL		AFL_new		AFLeg		AFLeg_new
Average accuracy	93.209±1.016		90.462±1.237		91.825±0.984		93.207±1.113		95.477±1.217		94.873±0.992		93.560±1.022		92.881±2.651		93.137±1.027		91.481±1.239		93.431±1.135		91.138±1.379		93.354±1.205		90.329±1.522		93.096±1.198
Agnostic loss	0.454±0.084		0.618±0.094		0.515±0.080		0.273±0.030		0.262±0.063		0.294±0.090		0.388±0.064		0.415±0.150		0.465±0.089		0.438±0.037		0.308±0.036		0.495±0.070		0.323±0.067		0.501±0.086		0.331±0.067
Accuracy parity	3.907±1.145		5.470±1.393		4.414±1.278		1.208±0.274		2.240±0.635		2.951±0.952		3.145±0.778		3.702±1.891		4.056±1.247		2.800±0.327		1.726±0.232		3.817±0.631		2.300±0.658		4.134±0.824		2.502±0.689
Average accuracy 95.47739009780555 94.87320320263896
Agnostic loss 0.262280584871769 0.27269178330898286
Accuracy parity 1.2077828065133935 1.7259845010068087


In [14]:
get_table('MNIST', 'partial_class_ni5', seeds, methods, test=False)

Index	FedAvg		qFFL		PropFair		FedMGDA		FedFV		Tche		STche		EPO		FERERO		ExcessMTL		AdaExcessMTL		AFL		AFL_new		AFLeg		AFLeg_new
Average accuracy	94.704±0.274		93.577±0.296		93.300±0.228		94.644±0.237		96.141±0.575		95.655±0.340		94.899±0.256		94.321±0.781		94.689±0.284		92.665±0.102		94.809±0.222		92.536±0.213		95.047±0.295		92.533±0.223		94.806±0.272
Agnostic loss	0.247±0.023		0.292±0.021		0.299±0.018		0.224±0.012		0.178±0.027		0.198±0.024		0.228±0.012		0.255±0.033		0.250±0.023		0.304±0.014		0.212±0.010		0.327±0.014		0.217±0.015		0.332±0.017		0.233±0.008
Accuracy parity	1.249±0.304		1.392±0.309		1.321±0.309		0.763±0.111		0.920±0.203		1.256±0.227		0.962±0.217		1.448±0.374		1.276±0.311		0.962±0.293		0.610±0.193		1.246±0.309		0.800±0.198		1.340±0.330		0.975±0.201
Average accuracy 96.14146087633713 95.65484557259546
Agnostic loss 0.17833960205316543 0.19780481308698655
Accuracy parity 0.6103811653765784 0.7632726320786095


In [15]:
get_table('CIFAR10', 'rotation', seeds, methods, test=False)

Index	FedAvg		qFFL		PropFair		FedMGDA		FedFV		Tche		STche		EPO		FERERO		ExcessMTL		AdaExcessMTL		AFL		AFL_new		AFLeg		AFLeg_new
Average accuracy	73.648±0.442		64.401±0.513		68.095±0.616		73.599±0.454		73.771±0.364		65.885±1.624		73.588±0.368		65.127±2.614		73.908±0.359		62.328±0.455		73.385±0.358		62.388±0.543		73.547±0.342		62.240±0.547		73.446±0.391
Agnostic loss	1.007±0.033		1.208±0.025		1.109±0.038		0.890±0.026		0.889±0.020		1.138±0.052		0.903±0.028		1.151±0.059		0.989±0.035		1.128±0.027		0.957±0.032		1.102±0.019		0.805±0.019		1.121±0.022		0.810±0.021
Accuracy parity	6.292±0.508		5.170±0.328		5.208±0.510		3.760±0.452		3.485±0.499		4.666±0.770		4.034±0.501		4.934±0.903		6.130±0.559		1.793±0.497		4.997±0.629		1.301±0.290		1.728±0.398		1.668±0.356		1.746±0.375
Average accuracy 73.9082 73.7712
Agnostic loss 0.8049812366962433 0.810417366027832
Accuracy parity 1.3010013250515275 1.6679361860420436


In [16]:
get_table('CIFAR10', 'partial_class_ni2', seeds, methods, test=False)

Index	FedAvg		qFFL		PropFair		FedMGDA		FedFV		Tche		STche		EPO		FERERO		ExcessMTL		AdaExcessMTL		AFL		AFL_new		AFLeg		AFLeg_new
Average accuracy	41.757±2.691		36.197±2.587		36.569±1.928		41.874±2.859		42.950±3.422		20.895±6.514		38.972±3.190		21.798±7.567		41.602±2.951		36.479±2.131		41.254±2.930		32.991±4.592		40.095±3.000		36.227±2.515		40.163±3.105
Agnostic loss	2.376±0.251		2.525±0.257		2.422±0.213		1.973±0.087		2.260±0.169		4.369±0.406		2.270±0.237		4.317±0.659		2.360±0.252		2.466±0.300		2.310±0.283		2.461±0.272		2.269±0.249		2.476±0.239		2.282±0.265
Accuracy parity	22.734±5.065		23.268±4.991		23.014±5.153		13.892±2.383		20.231±4.864		27.278±3.288		20.637±6.172		26.346±4.313		22.818±5.149		22.708±5.668		22.134±5.622		21.583±6.134		19.684±5.815		22.234±5.902		20.208±5.909
Average accuracy 42.95 41.87400000000001
Agnostic loss 1.9728360247612002 2.260126668214798
Accuracy parity 13.892124776767286 19.684011421357038


In [17]:
get_table('CIFAR10', 'partial_class_ni5', seeds, methods, test=False)

Index	FedAvg		qFFL		PropFair		FedMGDA		FedFV		Tche		STche		EPO		FERERO		ExcessMTL		AdaExcessMTL		AFL		AFL_new		AFLeg		AFLeg_new
Average accuracy	61.332±1.655		51.000±1.933		57.191±1.699		61.943±1.473		61.781±1.683		38.344±4.319		61.366±1.581		40.204±2.449		61.354±1.402		50.342±1.713		61.303±1.622		50.135±1.969		60.688±1.538		50.070±1.851		61.448±1.827
Agnostic loss	1.494±0.111		1.785±0.137		1.608±0.111		1.337±0.113		1.435±0.089		3.662±0.287		1.420±0.105		3.010±0.436		1.518±0.152		1.806±0.124		1.499±0.151		1.775±0.076		1.424±0.139		1.777±0.082		1.448±0.088
Accuracy parity	9.228±1.653		9.775±2.138		9.490±1.756		6.481±1.741		7.995±0.897		18.363±1.999		7.416±1.833		16.678±1.156		9.348±1.766		9.969±2.080		9.078±1.915		8.958±1.281		7.279±2.501		8.901±1.233		8.136±1.175
Average accuracy 61.943200000000004 61.7812
Agnostic loss 1.3366270780563354 1.4197010288238527
Accuracy parity 6.481112887183068 7.278557588034607
